In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

"""Version J: documented hybrid ensemble for EV-purchase prediction.

Purpose
-------
This script trains four gradient-boosting classifiers on a shared engineered
feature matrix and combines their predictions with an OOF-optimized probability
blend. It is the documented version of the Version J pipeline that achieved the
reported leaderboard score of 0.94625.

Pipeline summary
----------------
1. Read train, test, and sample-submission CSV files.
2. Convert known categorical values and the target to numeric representations.
3. Create income digits, commute digits, quantile bins, frequency encodings,
   arithmetic interactions, and six categorical interaction keys.
4. During every outer cross-validation fold, generate three leakage-safe target
   encodings with smoothing values auto, 10, and 100.
5. Train XGBoost, two LightGBM variants, and CatBoost for five random seeds and
   three stratified folds per seed.
6. Average fold predictions within each seed and then average across seeds.
7. Search all non-negative four-model blend weights in increments of 0.02.
8. Save the submission, OOF diagnostics, test predictions, blend weights, and
   every model-by-seed prediction.

Important methodological note
-----------------------------
Target encoders are fitted only on the training portion of each outer fold.
Validation and test rows are transformed afterward, which prevents target-label
leakage from the outer validation fold. Frequency encodings are computed from
the complete training feature distribution and do not use target labels.
"""

In [ ]:
import os
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
from xgboost import XGBClassifier


@dataclass
class Settings:
    """Central configuration for data access, validation, blending, and hardware.

    Attributes
    ----------
    target:
        Name of the binary response column.
    identifier:
        Row identifier copied to all output files but excluded from model input.
    splits:
        Number of outer stratified folds used for every random seed.
    seed:
        Global reproducibility seed and reference value for NumPy.
    path:
        Directory containing train.csv, test.csv, and sample_submission.csv.
    blend_step:
        Resolution of the exhaustive non-negative blend-weight grid.
    use_xgb_gpu, use_cat_gpu:
        Hardware switches for XGBoost and CatBoost.
    """
    target: str = "Will_Buy_EV"
    identifier: str = "id"
    splits: int = 3
    seed: int = 42
    path: str = "/kaggle/input/competitions/playground-series-s6e9/"
    blend_step: float = 0.02
    use_xgb_gpu: bool = True
    use_cat_gpu: bool = True


CFG = Settings()
SEEDS = [42, 0, 10, 155, 156]
SMOOTHINGS = [("auto", "auto"), ("10", 10.0), ("100", 100.0)]
np.random.seed(CFG.seed)

INTERACTION_SPECS = {
    "age_gender_x": ["Age", "Gender"],
    "age_city_x": ["Age", "City_Type"],
    "car_city_x": ["Current_Car_Type", "City_Type"],
    "charging_subsidy_x": ["Home_Charging_Possible", "Subsidy_Available"],
    "anxiety_commute_x": ["Range_Anxiety_Level", "commute_integer"],
    "income_city_x": ["inc_bin_600", "City_Type"],
}

In [ ]:

def read_data(directory):
    """Load the three required competition CSV files from one directory.

    Parameters
    ----------
    directory : str
        Directory searched for files whose names end in ``.csv``.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]
        Training data, test data, and the sample-submission template.

    Raises
    ------
    FileNotFoundError
        Raised when any required logical file name is absent.
    """
    # Store each CSV under its filename stem, for example ``train``.
    data = {}
    for filename in os.listdir(directory):
        match = re.fullmatch(r"(.+)\.csv", filename)
        if match:
            data[match.group(1)] = pd.read_csv(os.path.join(directory, filename))
    # Validate all inputs before feature engineering begins.
    missing = {"train", "test", "sample_submission"}.difference(data)
    if missing:
        raise FileNotFoundError(f"Missing CSV files in {directory}: {sorted(missing)}")
    return data["train"], data["test"], data["sample_submission"]


def add_digit(frame, source, position, output):
    """Extract one base-10 digit from a numeric column in place.

    A negative position extracts a decimal digit. Position ``-1`` extracts the
    tenths digit, position ``0`` the units digit, and position ``1`` the tens
    digit. Missing source values are treated as zero.

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame modified in place.
    source : str
        Name of the numeric source column.
    position : int
        Decimal position to extract.
    output : str
        Name assigned to the resulting int8 feature.
    """
    # Dividing by the positional scale moves the requested digit to the units
    # position; floor and modulo then isolate that digit.
    scale = 10.0 ** position
    frame[output] = (
        np.floor(frame[source].fillna(0).to_numpy(dtype=float) / scale) % 10
    ).astype("int8")


def add_features(train, test):
    """Create the complete Version J feature set for train and test data.

    The same deterministic transformations are applied to both datasets. Any
    operation that must learn distributional information, such as quantile bins
    or frequency maps, is fitted on training data and then applied to test data.

    Parameters
    ----------
    train, test : pd.DataFrame
        Raw competition datasets after removal of the identifier column.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        Independent transformed copies of train and test.
    """
    # Convert known string categories to compact numeric codes. The target is
    # converted from Yes/No to 1/0 in training data only.
    mapping = {
        "Gender": {"Male": 0, "Female": 1, "Other": 2},
        "City_Type": {"Suburban": 0, "Rural": 1, "Urban": 2},
        "Current_Car_Type": {"Sedan": 0, "SUV": 1, "Hatchback": 2, "Truck": 3},
        "Home_Charging_Possible": {"Yes": 1, "No": 0},
        "Subsidy_Available": {"No": 0, "Yes": 1},
        "Range_Anxiety_Level": {"Low": 0, "Medium": 1, "High": 2},
        "Will_Buy_EV": {"No": 0, "Yes": 1},
    }
    # Copy after replacement so the caller's DataFrames are not modified.
    train = train.replace(mapping).copy()
    test = test.replace(mapping).copy()

    # Original income digits.
    for k in [4, 3, 2, 1, 0]:
        for frame in (train, test):
            frame[f"inc_d{k}"] = (
                np.floor(frame["Annual_Income_USD"].fillna(0) / (10 ** k)) % 10
            ).astype("int8")

    # Selected broad-digit idea: focus on the missing commute decimal structure.
    for position, label in [(-2, "m2"), (-1, "m1"), (0, "p0"), (1, "p1")]:
        add_digit(train, "Daily_Commute_km", position, f"commute_digit_{label}")
        add_digit(test, "Daily_Commute_km", position, f"commute_digit_{label}")

    # Fit a high-resolution income quantile binning scheme on training data.
    kb = KBinsDiscretizer(n_bins=600, encode="ordinal", strategy="quantile")
    train["inc_bin_600"] = kb.fit_transform(train[["Annual_Income_USD"]]).ravel().astype("int32")
    test["inc_bin_600"] = kb.transform(test[["Annual_Income_USD"]]).ravel().astype("int32")

    # Build coarse income keys and unsupervised train-frequency encodings.
    for div, name in [(100, "income100_floor"), (1000, "income1000_floor")]:
        train[name] = np.floor(train["Annual_Income_USD"] / div).astype("int64")
        test[name] = np.floor(test["Annual_Income_USD"] / div).astype("int64")
        freq = train[name].value_counts(normalize=True)
        train[f"{name}_fe"] = train[name].map(freq).astype("float32")
        test[f"{name}_fe"] = test[name].map(freq).fillna(0).astype("float32")

    # Flooring commute distance creates a repeated key suitable for frequency
    # encoding and supervised target encoding inside each outer fold.
    train["commute_integer"] = np.floor(train["Daily_Commute_km"]).astype("int64")
    test["commute_integer"] = np.floor(test["Daily_Commute_km"]).astype("int64")
    freq = train["commute_integer"].value_counts(normalize=True)
    train["commute_integer_fe"] = train["commute_integer"].map(freq).astype("float32")
    test["commute_integer_fe"] = test["commute_integer"].map(freq).fillna(0).astype("float32")

    # Compact arithmetic interactions copied from the strong standalone concept.
    for frame in (train, test):
        home = frame["Charging_Stations_Near_Home"].astype(float)
        work = frame["Charging_Stations_Near_Work"].astype(float)
        income = frame["Annual_Income_USD"].astype(float)
        commute = frame["Daily_Commute_km"].astype(float)
        frame["station_total"] = home + work
        frame["station_difference"] = home - work
        frame["station_max"] = np.maximum(home, work)
        frame["station_min"] = np.minimum(home, work)
        frame["home_work_ratio"] = home / (work + 1.0)
        frame["income_per_station_home"] = income / (home + 1.0)
        frame["income_per_station_work"] = income / (work + 1.0)
        frame["commute_income_ratio"] = commute / (income / 1000.0 + 1.0)
        frame["income_log"] = np.log1p(income.clip(lower=0))

    # Version 1 interaction keys. Raw keys are dropped after fold TE.
    for name, columns in INTERACTION_SPECS.items():
        train[name] = train[columns].astype(str).agg("|".join, axis=1)
        test[name] = test[columns].astype(str).agg("|".join, axis=1)

    return train, test


In [ ]:

def make_models():
    """Construct the four unfitted estimators used by the ensemble.

    Returns
    -------
    dict[str, estimator]
        Ordered mapping containing XGBoost, deep LightGBM, shallow LightGBM,
        and CatBoost. The ordering is also used by the blend-weight search.

    Notes
    -----
    Random seeds are assigned later for each outer fold. Early stopping uses the
    corresponding outer validation partition.
    """
    # Primary XGBoost branch with conservative learning rate and regularization.
    xgb = XGBClassifier(
        max_depth=5, learning_rate=0.005,
        min_child_weight=4.103230762955265,
        subsample=0.6937319640790256,
        colsample_bytree=0.3525127902131756,
        gamma=1.0551020301589011,
        reg_lambda=0.06906555517077836,
        reg_alpha=0.0027899773248414774,
        early_stopping_rounds=500, n_estimators=40000,
        tree_method="hist", eval_metric="auc", objective="binary:logistic",
        device="cuda" if CFG.use_xgb_gpu else "cpu", n_jobs=-1, verbosity=0,
    )
    # High-capacity LightGBM branch intended to model fine interactions.
    lgb_deep = LGBMClassifier(
        objective="binary", metric="auc", n_estimators=40000,
        learning_rate=0.005, num_leaves=127, max_depth=-1,
        min_child_samples=20, min_child_weight=4.0,
        subsample=0.70, subsample_freq=1, colsample_bytree=0.35,
        reg_alpha=0.003, reg_lambda=0.07, max_bin=511,
        n_jobs=-1, verbosity=-1,
    )
    # Diverse shallow LightGBM regime from the uploaded pipeline.
    lgb_shallow = LGBMClassifier(
        objective="binary", metric="auc", n_estimators=20000,
        learning_rate=0.02, max_depth=5, num_leaves=32,
        min_child_samples=10, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.30, reg_alpha=0.071, reg_lambda=2.0,
        max_bin=1024, feature_pre_filter=False, min_split_gain=0.01,
        n_jobs=-1, verbosity=-1,
    )
    # CatBoost branch receives the same fully numeric feature matrix.
    cat_params = dict(
        loss_function="Logloss", eval_metric="AUC", iterations=40000,
        learning_rate=0.005, depth=8, l2_leaf_reg=3.0,
        random_strength=0.25, bootstrap_type="Bayesian",
        bagging_temperature=0.7, border_count=254, od_type="Iter",
        od_wait=500, use_best_model=True,
        task_type="GPU" if CFG.use_cat_gpu else "CPU",
        verbose=False, allow_writing_files=False,
    )
    if CFG.use_cat_gpu:
        cat_params["devices"] = "0"
    return {
        "xgb": xgb,
        "lgb_deep": lgb_deep,
        "lgb_shallow": lgb_shallow,
        "cat": CatBoostClassifier(**cat_params),
    }


In [ ]:

def triple_target_encode(X_train, X_valid, X_test, y_train, te_columns, raw_keys):
    """Add three leakage-safe target encodings for every selected column.

    Parameters
    ----------
    X_train, X_valid, X_test : pd.DataFrame
        Fold-specific matrices. They are modified and returned.
    y_train : pd.Series
        Labels from the outer training partition only.
    te_columns : list[str]
        Columns encoded with each smoothing configuration.
    raw_keys : list[str]
        String interaction keys removed after their numeric encodings are made.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]
        Numeric training, validation, and test matrices with the new encodings.
    """
    # Fit a separate encoder for each smoothing strength. ``fit_transform`` on
    # X_train uses the encoder's internal cross-fitting, while validation and
    # test data are transformed without access to their labels.
    for suffix, smooth in SMOOTHINGS:
        encoder = TargetEncoder(target_type="binary", smooth=smooth)
        names = [f"{c}_te_{suffix}" for c in te_columns]
        X_train[names] = encoder.fit_transform(X_train[te_columns].astype(str), y_train)
        X_valid[names] = encoder.transform(X_valid[te_columns].astype(str))
        X_test[names] = encoder.transform(X_test[te_columns].astype(str))
    # Remove raw string keys because all four estimators expect numeric input.
    X_train.drop(columns=raw_keys, inplace=True)
    X_valid.drop(columns=raw_keys, inplace=True)
    X_test.drop(columns=raw_keys, inplace=True)
    return X_train, X_valid, X_test


def fit_model(name, model, X_train, y_train, X_valid, y_valid, seed):
    """Clone, seed, and fit one estimator with model-specific API handling.

    Cloning guarantees that no fitted state is carried between folds. CatBoost,
    LightGBM, and XGBoost expose different random-seed and early-stopping APIs,
    so each family is handled explicitly.

    Returns
    -------
    estimator
        The fitted fold-specific model.
    """
    # Start from an unfitted copy of the configured base estimator.
    model = clone(model)
    if name == "cat":
        model.set_params(random_seed=seed)
        model.fit(X_train, y_train, eval_set=(X_valid, y_valid), verbose=False)
    elif name.startswith("lgb"):
        model.set_params(random_state=seed)
        model.fit(
            X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="auc",
            callbacks=[early_stopping(500, verbose=False), log_evaluation(0)],
        )
    else:
        model.set_params(random_state=seed)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    return model

In [ ]:

def triple_target_encode(X_train, X_valid, X_test, y_train, te_columns, raw_keys):
    """Add three leakage-safe target encodings for every selected column.

    Parameters
    ----------
    X_train, X_valid, X_test : pd.DataFrame
        Fold-specific matrices. They are modified and returned.
    y_train : pd.Series
        Labels from the outer training partition only.
    te_columns : list[str]
        Columns encoded with each smoothing configuration.
    raw_keys : list[str]
        String interaction keys removed after their numeric encodings are made.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]
        Numeric training, validation, and test matrices with the new encodings.
    """
    # Fit a separate encoder for each smoothing strength. ``fit_transform`` on
    # X_train uses the encoder's internal cross-fitting, while validation and
    # test data are transformed without access to their labels.
    for suffix, smooth in SMOOTHINGS:
        encoder = TargetEncoder(target_type="binary", smooth=smooth)
        names = [f"{c}_te_{suffix}" for c in te_columns]
        X_train[names] = encoder.fit_transform(X_train[te_columns].astype(str), y_train)
        X_valid[names] = encoder.transform(X_valid[te_columns].astype(str))
        X_test[names] = encoder.transform(X_test[te_columns].astype(str))
    # Remove raw string keys because all four estimators expect numeric input.
    X_train.drop(columns=raw_keys, inplace=True)
    X_valid.drop(columns=raw_keys, inplace=True)
    X_test.drop(columns=raw_keys, inplace=True)
    return X_train, X_valid, X_test


def fit_model(name, model, X_train, y_train, X_valid, y_valid, seed):
    """Clone, seed, and fit one estimator with model-specific API handling.

    Cloning guarantees that no fitted state is carried between folds. CatBoost,
    LightGBM, and XGBoost expose different random-seed and early-stopping APIs,
    so each family is handled explicitly.

    Returns
    -------
    estimator
        The fitted fold-specific model.
    """
    # Start from an unfitted copy of the configured base estimator.
    model = clone(model)
    if name == "cat":
        model.set_params(random_seed=seed)
        model.fit(X_train, y_train, eval_set=(X_valid, y_valid), verbose=False)
    elif name.startswith("lgb"):
        model.set_params(random_state=seed)
        model.fit(
            X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="auc",
            callbacks=[early_stopping(500, verbose=False), log_evaluation(0)],
        )
    else:
        model.set_params(random_state=seed)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    return model

In [ ]:

def run_cv(models, X, y, test, te_columns, raw_keys):
    """Run repeated stratified CV and collect OOF and test predictions.

    For each seed, every row appears in exactly one validation fold. Test
    predictions are averaged across folds within a seed. The returned arrays
    retain a separate column for every seed so seed-level diagnostics can be
    saved after training.

    Returns
    -------
    tuple[dict[str, np.ndarray], dict[str, np.ndarray]]
        OOF matrices and test-prediction matrices, each shaped as
        ``(number_of_rows, number_of_seeds)`` for every model.
    """
    # Preallocate one prediction matrix per model for deterministic assignment.
    oof = {n: np.zeros((len(X), len(SEEDS))) for n in models}
    pred = {n: np.zeros((len(test), len(SEEDS))) for n in models}
    # Repeat the entire stratified CV process under each split seed.
    for seed_pos, seed in enumerate(SEEDS):
        cv = StratifiedKFold(n_splits=CFG.splits, shuffle=True, random_state=seed)
        # Each row is validation data once per seed.
        for fold, (tr, va) in enumerate(cv.split(X, y), 1):
            Xt, Xv, Xs = X.iloc[tr].copy(), X.iloc[va].copy(), test.copy()
            yt, yv = y.iloc[tr], y.iloc[va]
            # Fit all supervised encodings exclusively on this fold's training rows.
            Xt, Xv, Xs = triple_target_encode(Xt, Xv, Xs, yt, te_columns, raw_keys)
            scores = {}
            # Fit every model on the identical fold-specific numeric matrices.
            for name, base in models.items():
                model = fit_model(name, base, Xt, yt, Xv, yv, seed + 100 * fold)
                pv = model.predict_proba(Xv)[:, 1]
                ps = model.predict_proba(Xs)[:, 1]
                oof[name][va, seed_pos] = pv
                pred[name][:, seed_pos] += ps / CFG.splits
                scores[name] = roc_auc_score(yv, pv)
            print(f"Seed {seed}, fold {fold}/{CFG.splits}: " +
                  str({k: round(v, 6) for k, v in scores.items()}))
    return oof, pred
    
def search_blend(arrays, y, step):
    """Exhaustively optimize four non-negative blend weights for OOF AUC.

    The integer-grid construction guarantees that all four weights sum to one.
    With a step of 0.02, there are 50 discrete weight units. This function is
    intentionally specialized to the four models returned by ``make_models``.

    Returns
    -------
    tuple[np.ndarray, float]
        Best four-element weight vector and its OOF ROC-AUC.
    """
    # Stack model predictions in the same ordering used by the caller.
    matrix = np.column_stack(arrays)
    units = int(round(1.0 / step))
    best_auc, best_w = -np.inf, None
    # Four models: only 23,426 simplex combinations at step 0.02.
    for a in range(units + 1):
        for b in range(units - a + 1):
            for c in range(units - a - b + 1):
                d = units - a - b - c
                w = np.array([a, b, c, d], dtype=float) / units
                auc = roc_auc_score(y, matrix @ w)
                if auc > best_auc:
                    best_auc, best_w = auc, w.copy()
    return best_w, best_auc

In [ ]:
"""Execute the complete Version J training, blending, and export workflow."""
# Step 1: Load all competition inputs and preserve identifiers for outputs.
train, test, submission = read_data(CFG.path)
train_ids, test_ids = train[CFG.identifier].copy(), test[CFG.identifier].copy()
# Step 2: Exclude the identifier from all feature engineering and models.
train = train.drop(columns=[CFG.identifier])
test = test.drop(columns=[CFG.identifier])
# Step 3: Build all deterministic Version J features.
train, test = add_features(train, test)
X = train.drop(columns=[CFG.target])
y = train[CFG.target].astype("int8")

# Step 4: Declare raw columns that receive three fold-specific target encodings.
base_te = [
    "Annual_Income_USD", "Daily_Commute_km", "inc_bin_600",
    "income1000_floor", "income100_floor", "commute_integer",
    "inc_d0", "inc_d1", "inc_d2", "inc_d3", "inc_d4",
    "commute_digit_m2", "commute_digit_m1", "commute_digit_p0", "commute_digit_p1",
    "Age", "Number_of_Cars_Owned", "Home_Charging_Possible",
    "Subsidy_Available", "Range_Anxiety_Level", "Gender",
    "Charging_Stations_Near_Work", "Charging_Stations_Near_Home",
    "Current_Car_Type", "City_Type", "station_total",
]
# These high-cardinality string keys are retained only until TE is complete.
raw_keys = list(INTERACTION_SPECS)
te_columns = base_te + raw_keys
missing = [c for c in te_columns if c not in X]
if missing:
    raise ValueError(f"Missing TE columns: {missing}")

print("VERSION J: interaction TE + triple TE + digits + arithmetic + diverse LGB")
# Step 5: Build estimators and run all 3-fold by 5-seed training jobs.
models = make_models()
oof_by_model, test_by_model = run_cv(models, X, y, test, te_columns, raw_keys)
# Step 6: Average each model's predictions across its five seeds.
oof_mean = {n: p.mean(axis=1) for n, p in oof_by_model.items()}
test_mean = {n: p.mean(axis=1) for n, p in test_by_model.items()}

print("\nIndividual five-seed OOF AUC values:")
for name, values in oof_mean.items():
    print(f"  {name}: {roc_auc_score(y, values):.6f}")

# Step 7: Optimize the final four-model probability blend on OOF AUC.
names = list(models)
weights, blend_auc = search_blend([oof_mean[n] for n in names], y, CFG.blend_step)
oof_blend = np.column_stack([oof_mean[n] for n in names]) @ weights
test_blend = np.column_stack([test_mean[n] for n in names]) @ weights

print("\nVersion J optimized weights:")
for name, weight in zip(names, weights):
    print(f"  {name}: {weight:.2f}")
print(f"Version J OOF AUC: {blend_auc:.6f}")
print(f"Delta versus original 0.946147: {blend_auc - 0.946147:+.6f}")
print(f"Delta versus Version 1 LB 0.94617: {blend_auc - 0.94617:+.6f} (OOF reference only)")

# Step 8: Write the leaderboard submission using sample-submission schema.
submission[CFG.target] = np.clip(test_blend, 1e-7, 1 - 1e-7)
submission.to_csv("submission.csv", index=False)
# Step 9: Save model-level OOF predictions for reproducibility and analysis.
pd.DataFrame({
    CFG.identifier: train_ids, "target": y.to_numpy(),
    **{f"oof_{n}": oof_mean[n] for n in names}, "oof_blend": oof_blend,
}).to_csv("oof_version_j.csv", index=False)
# Step 10: Save model-level test predictions and the final blend.
pd.DataFrame({
    CFG.identifier: test_ids,
    **{f"pred_{n}": test_mean[n] for n in names}, "test_blend": test_blend,
}).to_csv("test_predictions_version_j.csv", index=False)
pd.DataFrame({"model": names, "weight": weights}).to_csv(
    "blend_weights_version_j.csv", index=False
)

# Step 11: Save every model-by-seed prediction for future diagnostics.
seed_oof = pd.DataFrame({CFG.identifier: train_ids, "target": y.to_numpy()})
seed_test = pd.DataFrame({CFG.identifier: test_ids})
for name in names:
    for pos, seed in enumerate(SEEDS):
        seed_oof[f"{name}_seed_{seed}"] = oof_by_model[name][:, pos]
        seed_test[f"{name}_seed_{seed}"] = test_by_model[name][:, pos]
seed_oof.to_csv("oof_all_models_all_seeds_version_j.csv", index=False)
seed_test.to_csv("test_all_models_all_seeds_version_j.csv", index=False)
print("Saved all Version J outputs.")